# 🐍 Decoradores, Iteradores e Geradores em Python

Este notebook explora três conceitos avançados e fundamentais da linguagem Python.

---

## 📑 Índice

1. [Decoradores](#1-decoradores)
2. [Iteradores](#2-iteradores)
3. [Geradores](#3-geradores)
4. [Exercícios Práticos](#4-exercícios-práticos)

---

## 1. Decoradores

Um **decorador** é uma função que recebe outra função como argumento, adiciona alguma funcionalidade e retorna uma nova função.

> 💡 **Analogia**: imagine um decorador como uma **camada de envoltório** (wrapper) em torno de uma função original.

### 1.1 Conceito Básico — Função como Objeto

Em Python, funções são **objetos de primeira classe**: podem ser passadas como argumentos, retornadas por outras funções e atribuídas a variáveis.

In [ ]:
def saudacao(nome):
    return f'Olá, {nome}!'

# Funções são objetos
print(type(saudacao))

# Podemos atribuir a outra variável
cumprimentar = saudacao
print(cumprimentar('Maria'))

# Podemos passar como argumento
def executar(funcao, argumento):
    return funcao(argumento)

print(executar(saudacao, 'João'))

### 1.2 Primeiro Decorador Manual

Vamos criar um decorador que mede o tempo de execução de uma função.

In [ ]:
import time

def medir_tempo(funcao):
    """Decorador que mede o tempo de execução de uma função."""
    def wrapper(*args, **kwargs):
        inicio = time.time()
        resultado = funcao(*args, **kwargs)
        fim = time.time()
        print(f'⏱️  {funcao.__name__} executou em {fim - inicio:.6f}s')
        return resultado
    return wrapper


def somar_lenta(a, b):
    time.sleep(0.1)
    return a + b

# Aplicando o decorador manualmente
somar_lenta = medir_tempo(somar_lenta)
print(f'Resultado: {somar_lenta(10, 20)}')

### 1.3 Sintaxe `@` (Açúcar Sintático)

Python oferece a sintaxe `@decorador` para aplicar decoradores de forma mais elegante.

In [ ]:
@medir_tempo
def multiplicar_lenta(a, b):
    time.sleep(0.05)
    return a * b

print(f'Resultado: {multiplicar_lenta(5, 6)}')

### 1.4 Preservando Metadados com `functools.wraps`

Sem `wraps`, a função decorada perde seu nome, docstring e outros metadados.

In [ ]:
from functools import wraps

def meu_decorador(funcao):
    @wraps(funcao)  # ← preserva metadados!
    def wrapper(*args, **kwargs):
        """Wrapper interno."""
        print(f'Chamando {funcao.__name__}...')
        return funcao(*args, **kwargs)
    return wrapper

@meu_decorador
def calcular_area(base, altura):
    """Calcula a área de um retângulo."""
    return base * altura

print(calcular_area(10, 5))
print(f'Nome: {calcular_area.__name__}')
print(f'Docstring: {calcular_area.__doc__}')

### 1.5 Decoradores com Argumentos

Para criar decoradores que recebem parâmetros, precisamos de uma **fábrica de decoradores** (função que retorna um decorador).

In [ ]:
from functools import wraps

def repetir(vezes):
    """Fábrica de decoradores: repete a execução N vezes."""
    def decorador(funcao):
        @wraps(funcao)
        def wrapper(*args, **kwargs):
            resultados = []
            for i in range(vezes):
                print(f'  Execução {i+1}/{vezes}')
                resultados.append(funcao(*args, **kwargs))
            return resultados
        return wrapper
    return decorador

@repetir(vezes=3)
def lancar_dado():
    import random
    return random.randint(1, 6)

resultados = lancar_dado()
print(f'Resultados: {resultados}')

### 1.6 Múltiplos Decoradores

Podemos empilhar decoradores. A ordem de aplicação é **de baixo para cima**.

In [ ]:
def negrito(funcao):
    @wraps(funcao)
    def wrapper(*args, **kwargs):
        return f'**{funcao(*args, **kwargs)}**'
    return wrapper

def italico(funcao):
    @wraps(funcao)
    def wrapper(*args, **kwargs):
        return f'*{funcao(*args, **kwargs)}*'
    return wrapper

@negrito
@italico
def mensagem():
    return 'Python é incrível!'

# Equivalente a: negrito(italico(mensagem))
print(mensagem())

### 1.7 Decoradores de Classe

Decoradores também podem ser aplicados a métodos de classe.

In [ ]:
def log_acesso(metodo):
    @wraps(metodo)
    def wrapper(self, *args, **kwargs):
        print(f'[LOG] Acesso ao método {metodo.__name__} da classe {self.__class__.__name__}')
        return metodo(self, *args, **kwargs)
    return wrapper

class ContaBancaria:
    def __init__(self, titular, saldo):
        self.titular = titular
        self._saldo = saldo

    @property
    def saldo(self):
        return self._saldo

    @log_acesso
    def depositar(self, valor):
        self._saldo += valor
        return self._saldo

    @log_acesso
    def sacar(self, valor):
        if valor > self._saldo:
            raise ValueError('Saldo insuficiente')
        self._saldo -= valor
        return self._saldo

conta = ContaBancaria('Ana', 1000)
conta.depositar(500)
conta.sacar(200)

---

## 2. Iteradores

Um **iterador** é um objeto que implementa os métodos `__iter__()` e `__next__()`, permitindo percorrer uma sequência de valores um por um.

> 💡 **Analogia**: um iterador é como um **marcador de página** em um livro — ele lembra onde você parou e avança para o próximo item quando solicitado.

### 2.1 Protocolo do Iterador

Para ser um iterador, um objeto deve implementar:

- `__iter__()` → retorna o próprio objeto (`self`)
- `__next__()` → retorna o próximo valor ou levanta `StopIteration`

In [ ]:
class Contador:
    """Iterador que conta de 1 até N."""

    def __init__(self, limite):
        self.limite = limite
        self.atual = 0

    def __iter__(self):
        print('🔄 __iter__ chamado — reiniciando contador')
        self.atual = 0
        return self

    def __next__(self):
        self.atual += 1
        if self.atual > self.limite:
            print('🛑 StopIteration levantado')
            raise StopIteration
        print(f'  → __next__ retornou {self.atual}')
        return self.atual


contador = Contador(3)
print('Usando for:')
for numero in contador:
    print(f'  Recebido: {numero}')

print('\nUsando next() manualmente:')
contador2 = Contador(2)
iterador = iter(contador2)
print(next(iterador))
print(next(iterador))
try:
    print(next(iterador))
except StopIteration:
    print('Iterador esgotado!')

### 2.2 Iterável vs. Iterador

| Conceito | Definição | Métodos |
|----------|-----------|---------|
| **Iterável** | Objeto que pode ser percorrido | `__iter__()` |
| **Iterador** | Objeto que FAZ a iteração | `__iter__()` + `__next__()` |

Listas, strings, dicionários são **iteráveis**, mas não são iteradores. Chamamos `iter()` neles para obter um iterador.

In [ ]:
lista = [10, 20, 30]
print(f'Tipo da lista: {type(lista)}')
print(f'É iterável? {hasattr(lista, "__iter__")}')
print(f'É iterador? {hasattr(lista, "__next__")}')

it = iter(lista)  # cria um iterador a partir do iterável
print(f'\nTipo do iterador: {type(it)}')
print(f'É iterador? {hasattr(it, "__next__")}')

print(next(it))
print(next(it))
print(next(it))

### 2.3 Iterador Personalizado — Sequência de Fibonacci

Vamos criar um iterador que gera a sequência de Fibonacci até um limite.

In [ ]:
class Fibonacci:
    """Iterador que gera a sequência de Fibonacci até N termos."""

    def __init__(self, n_termos):
        self.n_termos = n_termos
        self.contador = 0
        self.a, self.b = 0, 1

    def __iter__(self):
        self.contador = 0
        self.a, self.b = 0, 1
        return self

    def __next__(self):
        if self.contador >= self.n_termos:
            raise StopIteration
        self.contador += 1
        valor = self.a
        self.a, self.b = self.b, self.a + self.b
        return valor


fib = Fibonacci(10)
print('Sequência de Fibonacci (10 termos):')
print(list(fib))

# Podemos iterar novamente porque __iter__ reinicia o estado
print('\nNovamente:')
print(list(fib))

---

## 3. Geradores

**Geradores** são uma forma simplificada de criar iteradores. Em vez de definir uma classe com `__iter__` e `__next__`, usamos funções com a palavra-chave **`yield`**.

> 💡 **Analogia**: um gerador é como um **cafeteira expresso** — prepara uma xícara por vez, sob demanda, em vez de fazer o café inteiro de uma vez.

### 3.1 Função Geradora com `yield`

Quando uma função contém `yield`, ela se torna um gerador. Cada chamada a `next()` executa o código até o próximo `yield` e pausa, preservando o estado local.

In [ ]:
def contar_ate(limite):
    """Gerador que conta de 1 até limite."""
    print('🚀 Gerador iniciado')
    for numero in range(1, limite + 1):
        print(f'  ⏸️  Pausando em yield, valor={numero}')
        yield numero
        print(f'  ▶️  Retomando após yield, valor={numero}')
    print('🏁 Gerador finalizado')


gen = contar_ate(3)
print(f'Tipo: {type(gen)}')
print(f'É iterador? {hasattr(gen, "__next__")}')

print('\n--- Chamando next() manualmente ---')
print(f'Resultado 1: {next(gen)}')
print(f'Resultado 2: {next(gen)}')
print(f'Resultado 3: {next(gen)}')

try:
    print(next(gen))
except StopIteration:
    print('Gerador esgotado!')

### 3.2 Gerador de Fibonacci (versão simplificada)

Compare com a versão usando classe (Iterador). O gerador é muito mais conciso.

In [ ]:
def fibonacci(n_termos):
    """Gerador da sequência de Fibonacci."""
    a, b = 0, 1
    for _ in range(n_termos):
        yield a
        a, b = b, a + b


print('Fibonacci com gerador (10 termos):')
print(list(fibonacci(10)))

print('\nFibonacci sob demanda:')
for valor in fibonacci(5):
    print(f'  {valor}')

### 3.3 Expressões Geradoras (Generator Expressions)

Sintaxe similar às list comprehensions, mas com `()` em vez de `[]`. São **preguiçosas** (lazy evaluation) — geram valores sob demanda, sem armazenar todos na memória.

In [ ]:
# List comprehension (eager — armazena TUDO na memória)
quadrados_lista = [x**2 for x in range(1_000_000)]
print(f'Lista: {type(quadrados_lista)}, tamanho na memória: ~{quadrados_lista.__sizeof__() / 1024 / 1024:.2f} MB')

# Generator expression (lazy — gera um por vez)
quadrados_gen = (x**2 for x in range(1_000_000))
print(f'Gerador: {type(quadrados_gen)}, tamanho na memória: ~{quadrados_gen.__sizeof__() / 1024:.2f} KB')

# Consumindo o gerador
soma = sum(quadrados_gen)
print(f'Soma dos quadrados: {soma}')

### 3.4 `yield from` — Delegando para outro iterável

`yield from` permite que um gerador delegue a iteração para outro gerador ou iterável, simplificando código aninhado.

In [ ]:
def sub_gerador(prefixo):
    for i in range(3):
        yield f'{prefixo}-{i}'

def gerador_principal():
    yield 'Início'
    # Sem yield from (manual e verboso):
    # for valor in sub_gerador('A'):
    #     yield valor
    
    # Com yield from (elegante):
    yield from sub_gerador('A')
    yield from sub_gerador('B')
    yield 'Fim'

print('Usando yield from:')
for item in gerador_principal():
    print(f'  {item}')

### 3.5 Geradores como Corrotinas — `send()` e Estado

Além de produzir valores, geradores podem **receber** valores externos via `send()`, funcionando como corrotinas (co-rotinas).

In [ ]:
def acumulador():
    """Corrotina que acumula valores enviados."""
    total = 0
    while True:
        valor = yield total  # recebe valor via send() e retorna total
        if valor is None:
            break
        total += valor


acc = acumulador()
next(acc)  # priming: avança até o primeiro yield

print(f'Enviando 10 → total: {acc.send(10)}')
print(f'Enviando 20 → total: {acc.send(20)}')
print(f'Enviando 5  → total: {acc.send(5)}')
acc.close()  # encerra o gerador
print('Gerador fechado.')

### 3.6 Comparativo: Memória e Performance

Vamos comparar o consumo de memória entre lista, iterador de classe e gerador ao processar grandes volumes.

In [ ]:
import sys

# 1. Lista (armazena todos os valores)
lista = list(range(1_000_000))
memoria_lista = sys.getsizeof(lista)
print(f'Lista: {memoria_lista:,} bytes ({memoria_lista/1024/1024:.2f} MB)')

# 2. Gerador (não armazena, gera sob demanda)
gen = range(1_000_000)  # range já é um objeto gerador-like
memoria_gen = sys.getsizeof(gen)
print(f'Range: {memoria_gen:,} bytes ({memoria_gen/1024:.2f} KB)')

# 3. Generator expression
gen_expr = (x for x in range(1_000_000))
memoria_expr = sys.getsizeof(gen_expr)
print(f'GenExpr: {memoria_expr:,} bytes ({memoria_expr/1024:.2f} KB)')

# Soma sem armazenar tudo na memória
resultado = sum(x for x in range(1_000_000) if x % 2 == 0)
print(f'\nSoma dos pares até 1M: {resultado}')

---

## 4. Exercícios Práticos

### Exercício 1: Decorador de Cache Simples

Crie um decorador `@cache` que armazena os resultados de chamadas anteriores e retorna o valor em cache se a função for chamada com os mesmos argumentos.

In [ ]:
from functools import wraps

def cache(funcao):
    """Decorador que memoiza resultados de chamadas."""
    memoria = {}
    
    @wraps(funcao)
    def wrapper(*args):
        if args not in memoria:
            print(f'  [CACHE MISS] Calculando {funcao.__name__}{args}')
            memoria[args] = funcao(*args)
        else:
            print(f'  [CACHE HIT]  Retornando valor em cache para {args}')
        return memoria[args]
    
    return wrapper


@cache
def fatorial(n):
    if n < 2:
        return 1
    return n * fatorial(n - 1)

print(fatorial(5))
print(fatorial(5))  # cache hit
print(fatorial(6))  # usa cache de fatorial(5)

### Exercício 2: Iterador Reverso

Implemente um iterador `Reverso` que percorre uma string de trás para frente, um caractere por vez.

In [ ]:
class Reverso:
    """Iterador que percorre uma string de trás para frente."""

    def __init__(self, texto):
        self.texto = texto
        self.indice = len(texto)

    def __iter__(self):
        self.indice = len(self.texto)
        return self

    def __next__(self):
        self.indice -= 1
        if self.indice < 0:
            raise StopIteration
        return self.texto[self.indice]


rev = Reverso('Python')
print(''.join(rev))  # nohtyP

# Usando em for
for char in Reverso('ABC'):
    print(char, end=' ')
print()

### Exercício 3: Gerador de Números Primos

Crie um gerador infinito que produza números primos, um por um, sob demanda.

In [ ]:
def primos():
    """Gerador infinito de números primos."""
    numero = 2
    primos_encontrados = []
    
    while True:
        eh_primo = True
        for p in primos_encontrados:
            if p * p > numero:
                break
            if numero % p == 0:
                eh_primo = False
                break
        
        if eh_primo:
            primos_encontrados.append(numero)
            yield numero
        
        numero += 1


gen_primos = primos()
print('Primeiros 15 números primos:')
print([next(gen_primos) for _ in range(15)])

print('\nPróximos 5:')
print([next(gen_primos) for _ in range(5)])

### Exercício 4: Pipeline com Geradores

Crie um pipeline de processamento de dados usando geradores: leitura → filtro → transformação.

In [ ]:
def leitura(dados):
    """Simula leitura de dados de uma fonte."""
    for item in dados:
        yield item

def filtro_pares(fonte):
    """Filtra apenas números pares."""
    for item in fonte:
        if item % 2 == 0:
            yield item

def transformar_quadrado(fonte):
    """Eleva ao quadrado."""
    for item in fonte:
        yield item ** 2

# Pipeline: leitura → filtro → transformação
dados = range(20)
pipeline = transformar_quadrado(filtro_pares(leitura(dados)))

print('Pipeline: pares ao quadrado de 0 a 19')
print(list(pipeline))

# Versão com generator expression (mais concisa)
pipeline_expr = (x**2 for x in range(20) if x % 2 == 0)
print(f'\nMesmo resultado: {list(pipeline_expr)}')

---

## ✅ Resumo dos Conceitos

| Conceito | Quando Usar | Sintaxe-Chave |
|----------|-------------|---------------|
| **Decorador** | Adicionar comportamento transversal (log, cache, validação) | `@decorador` |
| **Iterador** | Controle total sobre o estado da iteração | `__iter__`, `__next__` |
| **Gerador** | Sequências grandes ou infinitas, lazy evaluation | `yield` |
| **GenExpr** | Transformações simples em coleções grandes | `(x for x in ...)` |
| **`yield from`** | Delegar iteração para outro iterável | `yield from iteravel` |

---

> 🎓 **Dica Final**: Use **geradores** quando a memória for uma preocupação ou quando os dados forem potencialmente infinitos. Use **decoradores** para separar responsabilidades transversais (cross-cutting concerns) do código de negócio. Use **iteradores de classe** apenas quando precisar de estado complexo ou de métodos adicionais além da iteração.